In [34]:
id = "rajpurkar/squad"
from datasets import load_dataset

dataset = load_dataset(id, split="train[:5000]")

In [35]:
# Split train / test
training_set = dataset.train_test_split(test_size=0.2)

In [36]:
# Load toakenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [37]:
# Process data function
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)


        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:

            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [38]:
tokenized_dataset = training_set.map(
    preprocess_function,
    batched=True,
    remove_columns=training_set["train"].column_names
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [39]:
# Load model
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
# Build training batches
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [41]:
from transformers import TrainingArguments, Trainer

# Create training hyperparameters
training_args = TrainingArguments(
    output_dir="my_awesome_qa_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

# Create trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [42]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.519199
2,2.784180,1.809715
3,2.784180,1.717561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=750, training_loss=2.336892049153646, metrics={'train_runtime': 889.0473, 'train_samples_per_second': 13.498, 'train_steps_per_second': 0.844, 'total_flos': 1175877900288000.0, 'train_loss': 2.336892049153646, 'epoch': 3.0})

In [43]:
question = "What did the Virgin Mary reportedly do in 1858?"
context = (
    "Architecturally, the school has a Catholic character. Atop the Main "
    "Building's gold dome is a golden statue of the Virgin Mary. Immediately "
    "in front of the Main Building and facing it, is a copper statue of Christ "
    "with arms upraised with the legend \"Venite Ad Me Omnes\". Next to the "
    "Main Building is the Basilica of the Sacred Heart. Immediately behind the "
    "basilica is the Grotto, a Marian place of prayer and reflection. It is a "
    "replica of the grotto at Lourdes, France where the Virgin Mary reputedly "
    "appeared to Saint Bernadette Soubirous in 1858. At the end of the main "
    "drive (and in a direct line that connects through 3 statues and the Gold "
    "Dome), is a simple, modern stone statue of Mary."
)

In [44]:
inputs = tokenizer(question, context, return_tensors="pt").to(model.device)

In [45]:
import torch

with torch.no_grad():
    outputs = model(**inputs)

# Select highest probability start and end positions
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

# Predict
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)

'saint bernadette soubirous'